# 02 - Cleaning and merging (Transfermarkt dataset)

**Why this dataset:** football-data.org's free tier returned no market values and no contract dates (0 of 543 players), so market values come from the Transfermarkt-based dataset (`player-scores` on Kaggle), which also has minutes played, assists and international caps.

**Goal:** one row per Premier League player for the **2025/26 season**:
- target: the player's latest market value inside the season window
- features: age, position, minutes, goals and assists (per 90), international caps, contract years left

The notebook prints how many players are lost at each step ("funnel") so we can explain it in the presentation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed"); PROCESSED.mkdir(parents=True, exist_ok=True)

LEAGUE = "GB1"                          # Premier League's code in Transfermarkt data
SEASON_START = pd.Timestamp("2025-07-01")
SEASON_END = pd.Timestamp("2026-06-30")
MIN_MINUTES = 450                       # ignore players with very little playing time (per-90 stats get noisy)

## 1. Check the league code

In [ ]:
comps = pd.read_csv(RAW / "competitions.csv")
print(comps.columns.tolist())
comps[comps["competition_id"] == LEAGUE].T

## 2. Appearances -> season totals per player

In [ ]:
app = pd.read_csv(
    RAW / "appearances.csv",
    usecols=["player_id", "date", "competition_id", "goals", "assists",
             "minutes_played", "yellow_cards", "red_cards"],
)
app["date"] = pd.to_datetime(app["date"], errors="coerce")
print("all appearances in file:", len(app))

app = app[(app["competition_id"] == LEAGUE) & app["date"].between(SEASON_START, SEASON_END)]
assert len(app) > 0, "No Premier League appearances in the window - check LEAGUE code and dates"
print("PL appearances in 2025/26:", len(app), "| from", app["date"].min().date(), "to", app["date"].max().date())

In [ ]:
stats = (
    app.groupby("player_id")
       .agg(appearances=("date", "count"),
            minutes=("minutes_played", "sum"),
            goals=("goals", "sum"),
            assists=("assists", "sum"),
            yellow_cards=("yellow_cards", "sum"),
            red_cards=("red_cards", "sum"))
       .reset_index()
)
print("players with at least one PL appearance:", len(stats))
stats.sort_values("goals", ascending=False).head()

## 3. Player profile (position, birth date, caps, contract)

In [ ]:
players = pd.read_csv(
    RAW / "players.csv",
    usecols=["player_id", "name", "position", "sub_position", "date_of_birth",
             "international_caps", "contract_expiration_date", "current_club_name"],
)
players["date_of_birth"] = pd.to_datetime(players["date_of_birth"], errors="coerce")
players["contract_expiration_date"] = pd.to_datetime(players["contract_expiration_date"], errors="coerce")
print(players.shape)
players.isna().mean().round(3)

## 4. Market value = the player's latest valuation inside the season window

`player_valuations.csv` is a history of valuations. For each player we take the most recent one dated within 2025/26. Players with no valuation in that window are dropped (and counted below).

In [ ]:
val = pd.read_csv(RAW / "player_valuations.csv", usecols=["player_id", "date", "market_value_in_eur"])
val["date"] = pd.to_datetime(val["date"], errors="coerce")
val = val[val["date"].between(SEASON_START, SEASON_END)]
val = (
    val.sort_values("date")
       .groupby("player_id").tail(1)
       .rename(columns={"date": "valuation_date", "market_value_in_eur": "market_value"})
)
print("players with a valuation in the window:", len(val))
val["valuation_date"].describe()

## 5. Merge and build features

In [ ]:
funnel = {}
funnel["PL players with appearances"] = len(stats)

df = stats.merge(players, on="player_id", how="left")
funnel["after joining profile"] = len(df)

df = df.merge(val, on="player_id", how="inner")
funnel["with a valuation in 2025/26"] = len(df)

df = df[df["minutes"] >= MIN_MINUTES]
funnel[f">= {MIN_MINUTES} minutes"] = len(df)

df["age"] = (df["valuation_date"] - df["date_of_birth"]).dt.days / 365.25
df["contract_years_left"] = ((df["contract_expiration_date"] - df["valuation_date"]).dt.days / 365.25).clip(lower=0)
df["goals_per90"] = df["goals"] / df["minutes"] * 90
df["assists_per90"] = df["assists"] / df["minutes"] * 90
df["international_caps"] = df["international_caps"].fillna(0)

df = df.dropna(subset=["age", "position", "market_value"])
df = df[df["market_value"] > 0]
funnel["final (no missing age/position/value)"] = len(df)

pd.Series(funnel, name="players").to_frame()

### Sanity check

The most valuable players should look right (stars at the top). Missing values per column are shown after.

In [ ]:
cols = ["name", "current_club_name", "position", "age", "minutes", "goals", "assists",
        "goals_per90", "assists_per90", "international_caps", "contract_years_left",
        "valuation_date", "market_value"]
df = df[["player_id"] + cols].reset_index(drop=True)
display(df.sort_values("market_value", ascending=False).head(10))
print(df.isna().sum()[lambda s: s > 0])

In [ ]:
df.describe().T[["count", "mean", "min", "50%", "max"]].round(2)

## 6. Save

In [ ]:
out = PROCESSED / "PL_2025_26_model_data.csv"
df.to_csv(out, index=False)
print("saved", out, df.shape)

## Notes for the presentation

- The funnel above shows how many players survive each step.
- Contract years left is missing for some players; the model notebook fills those with the median.
- International caps is the latest snapshot in `players.csv`, not the value on the valuation date.
- The dataset snapshot ends in July 2026 (updates are paused), so this is a fixed 2025/26 sample.